### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output


### Pydantic

Pydantic models provide richest feature set with field validation, descriptions and nested structures

In [1]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, client=<groq.resources.chat.completions.Completions object at 0x00000255B0339940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255B033A660>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="Title of the the movie")
    year:int=Field(description="This is the year when movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="The movie's ratings out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, client=<groq.resources.chat.completions.Completions object at 0x00000255B0339940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255B033A660>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the the movie', 'type': 'string'}, 'year': {'description': 'This is the year when movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': "The movie's ratings out of 10", 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name

In [31]:
model.invoke("Provide details about movie Inception in 1 line")

AIMessage(content="Directed by Christopher Nolan, *Inception* follows a skilled thief who steals secrets from within the subconscious of his targets during dream-sharing, but is offered a chance at redemption to perform the opposite task: planting an idea in a CEO's mind.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 22, 'total_tokens': 72, 'completion_time': 0.097188373, 'completion_tokens_details': None, 'prompt_time': 0.001358192, 'prompt_tokens_details': None, 'queue_time': 0.048749997, 'total_time': 0.098546565}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1641d-a408-76a2-a1c8-da75b2cb1dbb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 50, 'total_tokens': 72})

In [5]:
model_with_structure.invoke("Provide details about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="Title of the the movie")
    year:int=Field(description="This is the year when movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="The movie's ratings out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide short details about movie Inception in 2 lines")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'mnh5xrbsa', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 360, 'total_tokens': 428, 'completion_time': 0.170670251, 'completion_tokens_details': None, 'prompt_time': 0.027211848, 'prompt_tokens_details': None, 'queue_time': 0.129578631, 'total_time': 0.197882099}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16411-29be-7d02-a625-3ace9cbb94d0-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'mnh5xrbsa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 360, 'output_tokens': 68, 'total

### Nested Structure

In [18]:
from pydantic import BaseModel, Field 

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None=Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide short details about movie Inception in 2 lines")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '14exd0nq6', 'function': {'arguments': '{"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"}],"genres":["Action","Sci-Fi","Thriller"],"title":"Inception","year":2010}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 393, 'total_tokens': 503, 'completion_time': 0.252816879, 'completion_tokens_details': None, 'prompt_time': 0.028547209, 'prompt_tokens_details': None, 'queue_time': 0.054043691, 'total_time': 0.281364088}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16414-c2ef-7cf3-927b-2abb3000f60b-0', tool_calls=[{'name': 'MovieDetails', 'args': {'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role'

### TypedDict

TypedDict provides simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [21]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"Title of the movie"]
    year: Annotated[int , ... , "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's ratings out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Provide details of movie Avengers")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [23]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None=Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide short details about movie Avengers")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'e1t661z80', 'function': {'arguments': '{"budget":220000000,"cast":[{"name":"Robert Downey Jr.","role":"Tony Stark / Iron Man"},{"name":"Chris Evans","role":"Steve Rogers / Captain America"},{"name":"Scarlett Johansson","role":"Natasha Romanoff / Black Widow"},{"name":"Chris Hemsworth","role":"Thor"},{"name":"Mark Ruffalo","role":"Dr. Bruce Banner / The Hulk"},{"name":"Jeremy Renner","role":"Clint Barton / Hawkeye"},{"name":"Tom Hiddleston","role":"Loki"}],"genres":["Action","Adventure","Science Fiction"],"title":"Avengers","year":2012}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 224, 'prompt_tokens': 570, 'total_tokens': 794, 'completion_time': 0.46215719, 'completion_tokens_details': None, 'prompt_time': 0.03939771, 'prompt_tokens_details': None, 'queue_time': 0.05385247, 'total_time': 0.5015549}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 

None


### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restriction. You can create it using the @dataclass decorator

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent

load_dotenv()  # Loads GOOGLE_API_KEY from .env

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="You are a helpful assistant.",
)

In [36]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description = "The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract content from: John Doe, john@example.com, (+91) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(+91) 123-4567'


In [37]:
result

{'messages': [HumanMessage(content='Extract content from: John Doe, john@example.com, (+91) 123-4567', additional_kwargs={}, response_metadata={}, id='fafb502c-3b40-4ea4-ba3b-9670b4ce067e'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(+91) 123-4567"}', 'extras': {'signature': 'EtkICtYIAWkUfRMwMZ+oMNegTepSSG5XqBTcB6mv9kUP21mBITcQ6K/ia6Vgux3oL4lGHR8VciG9cyR31FX78YiU4DsfdrTeC7xQ0JXGzLTH1rYVIvyecJ3iNtzS0WS1D08FIoO8x86sKUMwIGalHGCmMuPAN5d9GVnxbghZB74SjlsmIyI4MedAtQdEtXO5XrJot+d/tb3JOXbMNzGQWaNUmUAH9vRxjNEsZlDYEGCnSbv7EktmoU7I1Rz1YlF3dNR0hO8Ptdr67suaTRUOY7MmzrVI1HXp1KQrXDuvrBeHe17g4QLeNZ7JCrkD3mw0EVWxpKLOCCqFp3Vei9FU5UgPg3fb9sk6wkxAyU89KG0CRgPfOltWgXV0wxcPsqG0fWIiTy1t5xvgQqFeFlsRcBHUVUoPpkE4Zu86lPfhrvBy9KFlutnBMf+fFDaKzmxqQ8LTDKxpexbNScxl9dTPr6+PQiBFk0FvIGtTCEGtmLjiFpBxS6f+bCZFim330FERaPypxXI4vJHzg1h/1knMIRcHboU96yjUbeuqkSuzUrkY22F/W3X/YfdvW0zjgRSZAWm537ZAX+Yo3BsVN8HV1sjLZ6u6IrOLFIjLTHBoZ/sWEK0TwpLp+xTbrvck2J2fFo9EGtpSkZB9h/00JQ1n+QT

In [38]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str #The name of the person
    email: str #The email address of person
    phone: str #Phone number of person 

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract content from: John Doe, john@example.com, (+91) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(+91) 123-4567')
